# GP Kernel Zoo Tutorial

Purpose: compare kernel choices on trend + periodic + local bump signal.


## Learning Roadmap

- Observe how kernel assumptions shape mean fit and uncertainty.
- Compare metrics across kernels, not just visual fit quality.
- Identify which kernels are robust under mild OOD extrapolation.


In [ ]:
# Step 1: import dependencies and GP kernel classes
# Configure Python path for local package imports
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'gp' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import math
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

from deepuq.models import (
    GaussianProcessRegressor,
    LinearKernel,
    MaternKernel,
    PeriodicKernel,
    RBFKernel,
    RationalQuadraticKernel,
)


In [ ]:
# Step 2: define calibration metrics helper
def regression_metrics(y_true, mean, var):
    y_true = y_true.reshape(-1)
    mean = mean.reshape(-1)
    var = var.reshape(-1).clamp_min(1e-8)
    rmse = torch.sqrt(torch.mean((mean - y_true) ** 2)).item()
    nll = 0.5 * torch.mean(torch.log(2 * torch.pi * var) + (y_true - mean) ** 2 / var).item()
    std = torch.sqrt(var)
    lower = mean - 1.96 * std
    upper = mean + 1.96 * std
    coverage95 = torch.mean(((y_true >= lower) & (y_true <= upper)).float()).item()
    width95 = torch.mean(upper - lower).item()
    return {
        'rmse': rmse,
        'nll': nll,
        'coverage95': coverage95,
        'interval_width95': width95,
    }


In [ ]:
# Step 3: create trend + periodic + local-bump dataset
x_train = torch.linspace(-3.5, 3.5, 120).unsqueeze(-1)
true_fn = lambda x: 0.3 * x + 0.8 * torch.sin(2.3 * x) + 0.9 * torch.exp(-0.8 * (x - 0.6) ** 2)
y_train = true_fn(x_train) + 0.08 * torch.randn_like(x_train)

x_test = torch.linspace(-5, 5, 320).unsqueeze(-1)
y_test = true_fn(x_test)

kernels = {
    'RBF': RBFKernel(lengthscale=0.6, outputscale=1.2),
    'Matern-1.5': MaternKernel(lengthscale=0.7, outputscale=1.2, nu=1.5),
    'Matern-2.5': MaternKernel(lengthscale=0.7, outputscale=1.2, nu=2.5),
    'RQ': RationalQuadraticKernel(lengthscale=0.7, outputscale=1.1, alpha=0.8),
    'Periodic+Linear': PeriodicKernel(lengthscale=0.8, outputscale=0.9, period=2.7)
        + LinearKernel(variance=0.05),
}


In [ ]:
# Step 4: fit each kernel and collect metrics
results = {}
for name, kernel in kernels.items():
    gp = GaussianProcessRegressor(kernel=kernel, noise=0.008)
    gp.fit(x_train, y_train)
    uq = gp.predict_uq(x_test)
    results[name] = (uq, regression_metrics(y_test.squeeze(-1), uq.mean, uq.total_var))

for name, (_, m) in results.items():
    print(name, {k: round(v, 4) for k, v in m.items()})


In [ ]:
# Step 5: compare posterior curves per kernel
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
axes = axes.flatten()
for ax, (name, (uq, _)) in zip(axes, results.items()):
    std = torch.sqrt(uq.total_var)
    ax.scatter(x_train.numpy(), y_train.numpy(), s=8, alpha=0.25)
    ax.plot(x_test.numpy(), y_test.numpy(), 'k--', lw=1.0)
    ax.plot(x_test.numpy(), uq.mean.numpy(), lw=1.8)
    ax.fill_between(
        x_test.squeeze(-1).numpy(),
        (uq.mean - 1.96 * std).numpy(),
        (uq.mean + 1.96 * std).numpy(),
        alpha=0.18,
    )
    ax.set_title(name)
for ax in axes[len(results):]:
    ax.axis('off')
fig.suptitle('Kernel choice changes fit and uncertainty behavior')
plt.tight_layout()
plt.show()


In [ ]:
# Additional diagnostic: kernel metric comparison charts
# Visualizing metric tradeoffs makes model selection more systematic.
import pandas as pd

rows = []
for name, (_, m) in results.items():
    rows.append({'kernel': name, **m})
df = pd.DataFrame(rows).sort_values('nll')
display(df)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(df['kernel'], df['rmse'])
axes[0].set_title('RMSE (lower is better)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(df['kernel'], df['nll'])
axes[1].set_title('Gaussian NLL (lower is better)')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(df['kernel'], df['coverage95'])
axes[2].axhline(0.95, color='k', ls='--', lw=1)
axes[2].set_title('95% Coverage (target ~0.95)')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()
